# Raport końcowy: reprezentacje ukrytych kart development w Catanie

Ten notebook jest samodzielnym podsumowaniem projektu na przedmiot **Uczenie Reprezentacji**. Zbiera wymagane elementy z polecenia: opis problemu, danych, modeli, eksperymentów, implementacji, decyzji projektowych, rezultatów oraz ograniczeń.

Projekt bada, czy sekwencyjne modele reprezentacji potrafią nauczyć się stanu przekonań o typach ukrytych kart development na podstawie publicznie obserwowalnej historii gry w Catanie.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display, Image

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
RESULTS = ROOT / 'results'
FIGURES = RESULTS / 'figures'
DATA_SPLITS = ROOT / 'data' / 'splits'

pd.set_option('display.max_colwidth', 120)


def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f)


def show_missing(path, command=None):
    msg = f'Brak artefaktu: `{path}`.'
    if command:
        msg += f' Aby go odtworzyć, uruchom: `{command}`.'
    display(Markdown(msg))


artifacts = {
    'data/splits': DATA_SPLITS.exists(),
    'results/final_metrics.json': (RESULTS / 'final_metrics.json').exists(),
    'results/baseline_metrics.json': (RESULTS / 'baseline_metrics.json').exists(),
    'results/search_results.json': (RESULTS / 'search_results.json').exists(),
    'results/nb03_beta_sweep.json': (RESULTS / 'nb03_beta_sweep.json').exists(),
    'results/figures': FIGURES.exists(),
}

pd.DataFrame(
    [{'artefakt': k, 'dostępny lokalnie': 'tak' if v else 'nie'} for k, v in artifacts.items()]
)


## Opis problemu

**Dane wejściowe.** Wejściem są sekwencje obserwacji publicznego stanu gry z perspektywy jednego obserwowanego gracza. Jedna sekwencja odpowiada parze `(game_id, observed_color)` i jest posortowana po `action_index`. Wektor obserwacji zawiera wyłącznie informacje dostępne publicznie: sumy zasobów, liczbę kart development na ręce, publiczne punkty, zagrane karty, budynki, drogi, aktualnego gracza, typ akcji, pozycję złodzieja oraz historię zakupów/zagrań kart development.

**Dane wyjściowe.** Dla każdej trzymanej karty development tworzymy osobną próbkę klasyfikacyjną. Etykietą jest prawdziwy typ karty: `KNIGHT`, `VICTORY_POINT`, `ROAD_BUILDING`, `MONOPOLY`, `YEAR_OF_PLENTY`.

**Typ zadania.** Jest to 5-klasowa klasyfikacja per karta z użyciem wyuczonej reprezentacji sekwencji gry. W propozycji projektu zapisano wariant 4-klasowy, ale w implementacji rozdzieliliśmy `MONOPOLY` i `YEAR_OF_PLENTY`, bo są osobnymi typami kart w talii i mają różną semantykę gry.

**Hipoteza badawcza.** Zakładaliśmy, że RSSM, dzięki rekurencyjnemu agregowaniu historii gry i jawnej separacji stanu deterministycznego oraz stochastycznego, da lepszą reprezentację niepewności niż VAE-LSTM, szczególnie w późniejszych fazach gry. Oczekiwaliśmy też, że modele reprezentacji poprawią wynik względem prostych reguł opartych na wiedzy dziedzinowej.


## Dane i przygotowanie zbioru

Dane są generowane samodzielnie w symulatorze `catanatron`. Generator zapisuje dwie powiązane tabele parquet:

| Tabela | Jednostka wiersza | Rola |
|---|---|---|
| `timesteps` | krok gry z perspektywy obserwowanego gracza | wejście sekwencyjne dla VAE/RSSM |
| `card_samples` | pojedyncza karta trzymana w danym kroku | próbka klasyfikacyjna z etykietą typu karty |

Łączenie odbywa się po `(game_id, observed_color, action_index)`. Split jest wykonywany per gra, żeby uniknąć przecieku między train/val/test. Gry z MCTS przy stole trafiają wyłącznie do testu jako przypadek `unseen_mcts`, a część gier bez MCTS tworzy test `seen`.

Najważniejsze decyzje przygotowania danych:

| Decyzja | Uzasadnienie |
|---|---|
| Perspektywa jednego obserwowanego gracza | Model ma uczyć stan przekonań widoczny dla konkretnego gracza, nie pełny stan gry. |
| Relatywne indeksowanie `p0..p3` | `p0` zawsze oznacza obserwowanego gracza, więc modele dostają spójny układ cech. |
| Wykluczenie `y_*` i ukrytych punktów VP z wejścia | Zapobiega bezpośredniemu przeciekowi targetu. |
| Osobna tabela `card_samples` | Jedna karta w jednym kroku to jedna próbka klasyfikacyjna. |
| Ewaluacja na startach tur | To moment decyzyjny, w którym informacja o niezagraniu karty ma sens interpretacyjny. |
| Izolacja MCTS w teście | Pozwala sprawdzić generalizację na niewidziany styl gry. |


## Opis modeli

### VAE-LSTM / β-VAE

Model bazuje na rodzinie VAE: Kingma i Welling, *Auto-Encoding Variational Bayes* (2013). W naszym wariancie enkoderem jest LSTM działający po sekwencji obserwacji. Dla każdego kroku generowany jest latent `z_t`, a dekoder rekonstruuje wektor obserwacji. Funkcja straty ma postać rekonstrukcja + `β * KL`, z liniowym wygrzewaniem β, aby ograniczyć ryzyko posterior collapse.

Warianty:

- `vae`: latent Gaussowski, probe używa deterministycznego `mu_t`.
- `vae_cat`: latent kategoryczny, KL względem stałego, jednostajnego priora.

### RSSM

RSSM opieramy na pracy Hafner et al. 2019, *Learning Latent Dynamics for Planning from Pixels* / PlaNet. Stan latentny składa się z deterministycznego stanu rekurencyjnego `h_t` i stochastycznego `z_t`. Model uczy prior `p(z_t | h_t)` oraz posterior `q(z_t | h_t, x_t)`, a do probe trafia reprezentacja `[h_t || z_t]`.

Warianty:

- `rssm_gauss`: Gaussowski stan stochastyczny i uczony prior.
- `rssm_cat`: kategoryczny stan stochastyczny inspirowany DreamerV2, z KL balancing i free bits.

### Modele nadzorowane i dodatkowe SSL

W repozytorium są też warianty supervised `*_supA/B/C` oraz wcześniejsza ścieżka Transformer SSL (`InfoNCE`, `Barlow Twins`, `MAE`). W finalnej interpretacji trzeba je traktować jako dodatkowe punkty odniesienia, a nie ten sam protokół co zamrożony encoder + linear probe.


## Opis eksperymentów

Główny protokół ewaluacyjny:

1. Uczymy reprezentację sekwencji na `timesteps`.
2. Zamrażamy encoder.
3. Dla każdej próbki-karty pobieramy embedding kroku i doklejamy jawne cechy per karta: `rounds_held`, `card_slot`, `n_hidden_cards`, `bought_at_action`, `current_rel_pos`.
4. Trenujemy głowicę/probe do klasyfikacji typu karty.
5. Raportujemy wyniki na TEST, osobno dla `seen`, `unseen_mcts` i `observed_type`.

Metryki:

- **macro-F1** jako metryka główna, ponieważ klasy są silnie niezbalansowane.
- F1 per klasa, aby pokazać zachowanie na rzadkich typach kart.
- Accuracy jako metryka pomocnicza.
- Przedziały ufności po seedach, gdy dostępne są wielokrotne uruchomienia.

Baseline końcowy to `src.baseline_heuristic`: nieuczony system reguł startujący od priora składu talii i aktualizujący go na podstawie jawnych sygnałów z gry. Logistic regression z EDA traktujemy jako pomocniczy baseline analityczny, nie jako główny baseline końcowy.


In [ ]:
commands = pd.DataFrame([
    {
        'krok': 'Generowanie danych',
        'komenda': 'uv run generate_dataset_v2.py --num 10000 --out-dir data --workers 16 --chunk-size 1000',
        'wynik': 'data/timesteps_*.parquet, data/card_samples_*.parquet',
    },
    {
        'krok': 'Weryfikacja danych',
        'komenda': 'uv run verify_dataset.py --data-dir data',
        'wynik': 'kontrole integralności i braku przecieku',
    },
    {
        'krok': 'Split',
        'komenda': 'uv run split_dataset.py --data-dir data --out-dir data/splits',
        'wynik': 'train/val/test parquet',
    },
    {
        'krok': 'Search hiperparametrów',
        'komenda': 'uv run python -m src.search --subsample-games 1500 --epochs 8 --probe-games 1200',
        'wynik': 'results/search_results.json',
    },
    {
        'krok': 'Finalny trening',
        'komenda': 'uv run python -m src.train_final --families all --epochs 25 --seeds 0,1,2',
        'wynik': 'results/final_metrics.json, checkpointy final_*_seed*.pt',
    },
    {
        'krok': 'Baseline heurystyczny',
        'komenda': 'uv run python -m src.baseline_heuristic',
        'wynik': 'results/baseline_metrics.json',
    },
])
commands


## Badanie hiperparametru β

Polecenie wymaga sprawdzenia wpływu co najmniej jednego hiperparametru metody. W projekcie wykorzystujemy sweep β w β-VAE, ponieważ β kontroluje kompromis między rekonstrukcją obserwacji a regularizacją przestrzeni latentnej. Zbyt małe β grozi autoenkoderem bez użytecznej struktury latentnej, a zbyt duże β nadmiernie ogranicza pojemność reprezentacji.


In [ ]:
beta_path = RESULTS / 'nb03_beta_sweep.json'
beta_results = load_json(beta_path)
if beta_results is None:
    show_missing(beta_path, 'uruchom sekcję §10 w notebooks/03_vae_rssm.ipynb')
else:
    df_beta = pd.DataFrame(beta_results)
    display(df_beta)
    if {'beta', 'f1_all', 'recon', 'kl'}.issubset(df_beta.columns):
        ax = df_beta.plot(x='beta', y=['f1_all', 'f1_seen', 'f1_un'], marker='o', logx=True, figsize=(7, 4))
        ax.set_xlabel('β')
        ax.set_ylabel('macro-F1')
        ax.set_title('Wpływ β w β-VAE na jakość downstream probe')


## Wyniki główne

Poniższa komórka ładuje finalne metryki, jeśli zostały wygenerowane. Jeśli artefaktów nie ma lokalnie, notebook pozostaje czytelny i wskazuje, które komendy trzeba uruchomić.


In [ ]:
def rows_from_final_metrics(data, source):
    rows = []
    if not data or 'final' not in data:
        return rows
    for model, payload in data['final'].items():
        summary = payload.get('summary', {})
        all_metrics = summary.get('all') or {}
        seen_metrics = summary.get('seen') or {}
        unseen_metrics = summary.get('unseen_mcts') or {}
        rows.append({
            'model': model,
            'źródło': source,
            'typ': 'supervised' if '_sup' in model else ('heuristic' if model == 'heuristic' else 'representation'),
            'macro_f1_all': all_metrics.get('mean'),
            'ci95_all': all_metrics.get('ci95'),
            'macro_f1_seen': seen_metrics.get('mean'),
            'macro_f1_unseen_mcts': unseen_metrics.get('mean'),
            'n_seeds': all_metrics.get('n_seeds'),
        })
    return rows

final_metrics = load_json(RESULTS / 'final_metrics.json')
baseline_metrics = load_json(RESULTS / 'baseline_metrics.json')
rows = rows_from_final_metrics(final_metrics, 'train_final') + rows_from_final_metrics(baseline_metrics, 'baseline_heuristic')

if not rows:
    show_missing(RESULTS / 'final_metrics.json', 'uv run python -m src.train_final --families all --epochs 25 --seeds 0,1,2')
    show_missing(RESULTS / 'baseline_metrics.json', 'uv run python -m src.baseline_heuristic')
else:
    df_results = pd.DataFrame(rows).drop_duplicates(subset=['model'], keep='last')
    df_results = df_results.sort_values('macro_f1_all', ascending=False, na_position='last')
    display(df_results)


In [ ]:
figure_files = [
    'rep_ranking.png',
    'rep_observed_type.png',
    'rep_perclass.png',
    'rep_confusion.png',
    'nb03_beta_sweep.png',
]

shown = False
for name in figure_files:
    path = FIGURES / name
    if path.exists():
        display(Markdown(f'### `{name}`'))
        display(Image(filename=str(path)))
        shown = True
if not shown:
    show_missing(FIGURES, 'uruchom notebooks/03_vae_rssm.ipynb oraz notebooks/04_final_report.ipynb po wygenerowaniu results/')


## Interpretacja wyników i hipotezy

Wyniki należy interpretować ostrożnie, bo finalna ewaluacja modeli uczonych korzysta z `eval_seq_len=512`, co może ograniczać pokrycie późnej fazy gry. To jest szczególnie ważne dla naszej hipotezy, która dotyczyła przewagi RSSM w agregowaniu długiej historii.

Najważniejsze wnioski projektowe:

- Hipoteza o jednoznacznej przewadze RSSM nad VAE nie została silnie potwierdzona. Różnice między rodzinami VAE/RSSM są małe i zależne od wariantu latentnego oraz protokołu ewaluacji.
- Heurystyka jest mocnym punktem odniesienia, bo wykorzystuje zasady gry, których modele muszą nauczyć się pośrednio z sekwencji.
- Rzadkie klasy `MONOPOLY` i `YEAR_OF_PLENTY` pozostają najtrudniejsze, co jest zgodne z dużą nierównowagą klas.
- Modele uczone gorzej generalizują na `MCTS`, jeśli ten styl był izolowany w teście.
- Warianty supervised mogą osiągać lepsze wyniki, ale nie są tym samym eksperymentem co zamrożona reprezentacja + linear probe.

Ponieważ dataset jest własny, nie raportujemy zewnętrznego state of the art. Zamiast tego porównujemy do baseline’u heurystycznego i prostszych punktów odniesienia.


## Decyzje projektowe

| Obszar | Decyzja | Konsekwencja |
|---|---|---|
| Definicja targetu | 5 klas zamiast 4 | Pełniejszy opis talii Catan; trudniejsze rzadkie klasy. |
| Jednostka próbki | pojedyncza karta w konkretnym kroku | Możemy oceniać F1 per typ karty. |
| Reprezentacja wejścia | tylko publiczne cechy, bez `y_*` | Brak przecieku targetu. |
| Split | per gra, MCTS izolowany w teście | Testuje generalizację na niewidziany styl. |
| Metryka główna | macro-F1 | Nie premiuje dominującej klasy kosztem rzadkich kart. |
| Baseline końcowy | nieuczony baseline heurystyczny | Interpretowalne odniesienie dla modeli reprezentacji. |
| VAE | LSTM + β-VAE | Prosty sekwencyjny bottleneck do porównania z RSSM. |
| RSSM | GRU `h_t` + stochastyczne `z_t` | Model jawnie uczy dynamikę stanu przekonań. |
| Hiperparametr | sweep β dla VAE | Spełnia wymaganie badania wpływu hiperparametru. |
| Prezentacja wyników | oddzielenie SSL/probe od supervised | Unika mylenia różnych protokołów uczenia. |


## Zgodność z poleceniem

| Wymaganie z polecenia | Gdzie jest adresowane w tym notebooku |
|---|---|
| Opis danych wejściowych | Sekcje „Opis problemu” i „Dane i przygotowanie zbioru” |
| Opis danych wyjściowych | Sekcja „Opis problemu” |
| Typ problemu | Sekcja „Opis problemu” |
| Hipoteza badawcza | Sekcje „Opis problemu” oraz „Interpretacja wyników i hipotezy” |
| Nazwa modelu | Sekcja „Opis modeli” |
| Publikacja metody | Sekcja „Opis modeli” |
| Zwięzły opis działania modelu | Sekcja „Opis modeli” |
| Różnice względem metody bazowej | Sekcja „Opis modeli” |
| Przygotowanie zbioru danych | Sekcja „Dane i przygotowanie zbioru” |
| Split train/val/test | Sekcja „Dane i przygotowanie zbioru” |
| Metryki | Sekcja „Eksperymenty” |
| Badanie hiperparametru | Sekcja „Badanie hiperparametru β” |
| Baseline dla własnego zbioru | Sekcje „Eksperymenty” i „Wyniki główne” |
| Kod w repozytorium | Sekcja z komendami uruchomieniowymi |
| Pliki zależności | `pyproject.toml` i `uv.lock`; środowisko uruchamiane przez `uv sync` / `uv run` |
| Kod eksperymentów i skrypty | Sekcja z komendami uruchomieniowymi |
| Oznaczenie nieautorskich fragmentów | Użyto `catanatron` jako zewnętrznego symulatora; modele zaimplementowane lokalnie na podstawie publikacji. |
| Czy wyniki potwierdzają hipotezę | Sekcja „Interpretacja wyników i hipotezy” |
| SOTA lub metoda naiwna | Brak SOTA dla własnego datasetu; porównanie do baseline’u heurystycznego. |
| Tabela wyników głównego eksperymentu | Sekcja „Wyniki główne”, po wygenerowaniu `results/final_metrics.json` |
| Tabela/wykres hiperparametrów | Sekcja „Badanie hiperparametru β” |
| Wkład zespołu | Sekcja poniżej |


## Wkład zespołu

- Wspólnie: przygotowanie pipeline’u danych, reprezentacji obserwacji, splitu, metryk i baseline’u.
- Osoba 1: VAE-LSTM oraz badanie wpływu β.
- Osoba 2: RSSM inspirowany Hafner et al. 2019.
- Wspólnie: ewaluacja porównawcza, analiza błędów i interpretacja wyników.

Jeśli notebook jest oddawany imiennie, warto przed oddaniem zastąpić „Osoba 1/Osoba 2” nazwiskami członków zespołu.


## Ograniczenia i dalsze prace

- `eval_seq_len=512` ogranicza pełne testowanie hipotezy o późnej fazie gry; najlepiej powtórzyć ekstrakcję embeddingów z dłuższym limitem lub streamingowym RSSM.
- Rzadkie klasy wymagają większej liczby gier albo metod przeciwdziałania ekstremalnej nierównowadze.
- Wyniki supervised i SSL/probe należy raportować osobno, bo odpowiadają na różne pytania.
- Baseline heurystyczny jest silny i interpretowalny, ale ma ograniczoną zdolność rozpoznawania rzadkich kart jako argmax.
- Brak publicznego benchmarku oznacza, że porównania są wewnątrzprojektowe.


## Propozycja 5-minutowej prezentacji

1. Problem: ukryte karty development jako stan przekonań.
2. Dane: dwie tabele, brak przecieku, split z MCTS w teście.
3. Modele: VAE-LSTM kontra RSSM.
4. Eksperyment: frozen encoder + probe, macro-F1, baseline heurystyczny.
5. Wyniki: ranking, F1 per klasa, generalizacja na MCTS.
6. Wniosek: RSSM nie daje jednoznacznej przewagi; reguły gry są bardzo mocnym sygnałem, a pełne sprawdzenie późnej gry wymaga dłuższej ewaluacji sekwencji.
